# Notebook 02: Data Cleaning, CPCB NAQI Calculation & Master Integration
**Project:** Exploratory Data Analysis of Air Pollution in Indian Cities
**Course:** IT 462 — Exploratory Data Analysis | Autumn Semester
**Mentor:** Dr. Gopinath Panda

This notebook implements:
1. Sensor outlier filtering and physical plausibility ceilings.
2. Regulatory CPCB National Air Quality Index (NAQI) sub-index calculation.
3. A 10% masking imputation experiment benchmarking Forward Fill, Linear Interpolation, KNN, and MICE.
4. Multi-source data integration into the 7-sheet consolidated Excel workbook.

In [1]:
import os, sys
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parents[0] if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import TABLES_DIR, PROCESSED_DATA_DIR, INTERIM_DATA_DIR
print("Project Root:", PROJECT_ROOT)

Project Root: /Users/Tanvi/Desktop/air_quality_eda


## 1. Imputation 10% Masking Benchmark Results
As required by the Project Blueprint, 10% of observed PM2.5 readings were randomly held out to benchmark imputation fidelity.

In [2]:
df_mask = pd.read_csv(TABLES_DIR / "imputation_masking_experiment.csv")
df_mask

,Method,Type,Evaluated_Points,RMSE,MAE
0,Forward Fill (7-day cap),Univariate Autoregressive,430,18.63,9.07
1,Linear Interpolation (7-day cap),Univariate Temporal,430,11.72,6.25
2,"K-Nearest Neighbors (KNN, k=5)",Multivariate Stoichiometric,430,49.75,26.34
3,"Iterative Imputer (MICE, BayesianRidge)",Multivariate Regression,430,16.10,8.00


## 2. Official CPCB NAQI Calculation & Distribution
Enforcing CPCB validity rules: >= 16 valid hours and >= 3 criteria pollutants (including at least one particulate matter).

In [3]:
df_aq = pd.read_csv(INTERIM_DATA_DIR / "aq_daily.csv")
print("AQI Categories Breakdown:")
print(df_aq["AQI_category"].value_counts())
df_aq[["city", "date", "PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "AQI", "AQI_category", "dominant_pollutant"]].head(10)

AQI Categories Breakdown:
AQI_category
Satisfactory                    1831
Moderate                        1464
Good                             682
Poor                             189
Very Poor                        114
No PM Indicator Present           38
Severe                            24
Insufficient Pollutants (<3)       6
Name: count, dtype: int64


,city,date,PM2.5,PM10,NO2,SO2,CO,O3,AQI,AQI_category,dominant_pollutant
0,Ahmedabad,2025-01-01,46.25,97.83,28.63,6.11,0.59,31.30,97.8,Satisfactory,PM10
1,Ahmedabad,2025-01-02,48.55,102.41,33.40,8.97,0.79,31.95,101.9,Moderate,PM10
2,Ahmedabad,2025-01-03,67.45,138.91,40.99,15.87,0.97,27.91,126.2,Moderate,PM10
3,Ahmedabad,2025-01-04,66.65,137.54,46.18,13.35,0.84,32.05,125.3,Moderate,PM10
4,Ahmedabad,2025-01-05,55.54,115.32,30.45,11.99,0.55,38.99,110.5,Moderate,PM10
5,Ahmedabad,2025-01-06,50.72,105.89,28.52,12.38,0.49,38.68,104.3,Moderate,PM10
6,Ahmedabad,2025-01-07,49.66,106.77,30.20,13.92,0.61,36.00,104.8,Moderate,PM10
7,Ahmedabad,2025-01-08,53.92,109.71,34.74,9.66,0.54,32.47,106.8,Moderate,PM10
8,Ahmedabad,2025-01-09,53.41,113.42,35.61,13.04,0.61,36.71,109.3,Moderate,PM10
9,Ahmedabad,2025-01-10,58.53,126.97,37.35,17.52,0.74,37.16,118.3,Moderate,PM10


## 3. Merged Master Dataset Overview

In [4]:
df_master = pd.read_csv(PROCESSED_DATA_DIR / "master.csv")
print(f"Master panel dimensions: {df_master.shape}")
df_master.head()

Master panel dimensions: (4348, 50)


,city,date,active_stations,valid_hours,PM2.5,PM10,NO2,SO2,CO,O3,...,month_cos,is_winter,is_monsoon,is_summer,is_post_monsoon,state,region,latitude,longitude,pm25_to_pm10_ratio
0,Ahmedabad,2025-01-01,9,24.0,46.25,97.83,28.63,6.11,0.59,31.30,...,0.866025,1,0,0,0,Gujarat,West,23.022846,72.60188,0.472759
1,Ahmedabad,2025-01-02,9,24.0,48.55,102.41,33.40,8.97,0.79,31.95,...,0.866025,1,0,0,0,Gujarat,West,23.022846,72.60188,0.474075
2,Ahmedabad,2025-01-03,9,24.0,67.45,138.91,40.99,15.87,0.97,27.91,...,0.866025,1,0,0,0,Gujarat,West,23.022846,72.60188,0.485566
3,Ahmedabad,2025-01-04,9,24.0,66.65,137.54,46.18,13.35,0.84,32.05,...,0.866025,1,0,0,0,Gujarat,West,23.022846,72.60188,0.484586
4,Ahmedabad,2025-01-05,9,24.0,55.54,115.32,30.45,11.99,0.55,38.99,...,0.866025,1,0,0,0,Gujarat,West,23.022846,72.60188,0.481616


## 4. Verification of Consolidated Excel Workbook

In [5]:
import openpyxl
wb = openpyxl.load_workbook(TABLES_DIR / "air_quality_master.xlsx", read_only=True)
print("Workbook Sheets:", wb.sheetnames)
wb.close()

Workbook Sheets: ['cities', 'aq_daily', 'weather_daily', 'fires_daily', 'calendar', 'master', 'metadata']
